# Avaliação Econométrica e Simulação de Política de Precificação

> **AVISO MANDATÓRIO DE COMPLIANCE E GOVERNANÇA:**  
> *Os resultados financeiros e operacionais apresentados neste notebook são **estritamente simulados em ambiente computacional controlado**, gerados a partir de processos estocásticos para validação do pipeline analítico do Copiloto de Precificação Inteligente.*

In [ ]:
from datetime import date
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tabulate import tabulate

from data_generator import SyntheticDataGenerator, VERTICAL_CONFIGS
from optimizer import calculate_optimal_price

# Fixação estrita de semente aleatória
SEED = 42
np.random.seed(SEED)

## 1. Geração e Split Temporal (Out-of-Time)

Para reproduzir as condições reais de produção sem vazamento de dados (*data leakage*), dividimos a série temporal de 365 dias em:
- **Treino:** 80% dias iniciais (~292 dias)
- **Teste (OOT):** 20% dias finais (~73 dias)

In [ ]:
generator = SyntheticDataGenerator(seed=SEED)
df_sales, _ = generator.generate_dataset(start_date=date(2023, 1, 1), days=365)

unique_dates = sorted(df_sales["sold_on"].unique())
split_idx = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_idx]

df_train = df_sales[df_sales["sold_on"] < split_date].copy()
df_test = df_sales[df_sales["sold_on"] >= split_date].copy()

print(f"Corte temporal de treino/teste em: {split_date}")
print(f"Tamanho do conjunto de treino: {len(df_train)} observações")
print(f"Tamanho do conjunto de teste:  {len(df_test)} observações")

## 2. Calibração da Elasticidade e Erro de Estimação

Avaliamos o desvio absoluto entre a elasticidade estimada $(\hat{\eta})$ e o parâmetro gerador real $(\eta_{\text{real}})$.

In [ ]:
df_train["ln_q"] = np.log(df_train["quantity"])
df_train["ln_p"] = np.log(df_train["unit_price"])
df_train["ln_p_comp"] = np.log(df_train["competitor_median_price"])

dow_dummies = pd.get_dummies(df_train["day_of_week"], prefix="dow", dtype=float)
selected_dow = [c for c in dow_dummies.columns if c != "dow_2"]
df_train = pd.concat([df_train, dow_dummies[selected_dow]], axis=1)

estimated_params = {}
summary_table = []

for vertical, config in VERTICAL_CONFIGS.items():
    sub = df_train[df_train["vertical"] == vertical]
    X = sm.add_constant(sub[["ln_p", "ln_p_comp"] + selected_dow])
    y = sub["ln_q"]
    res = sm.OLS(y, X).fit(cov_type="HC3")
    
    eta_hat = float(res.params["ln_p"])
    beta_hat = float(res.params["ln_p_comp"])
    eta_true = config.true_elasticity
    abs_error = abs(eta_hat - eta_true)
    
    estimated_params[vertical] = eta_hat
    summary_table.append([
        vertical,
        f"{eta_true:.2f}",
        f"{eta_hat:.4f}",
        f"{abs_error:.4f}",
        f"{res.rsquared:.4f}",
        f"{res.pvalues['ln_p']:.2e}"
    ])

print(tabulate(
    summary_table,
    headers=["Vertical", "η Real", "η Estimado", "Erro Absoluto |η̂ - η|", "R²", "p-valor"],
    tablefmt="github"
))

## 3. Simulação Comparativa de Lucro Acumulado no Conjunto de Teste

Comparamos o lucro acumulado $\sum (p - c) \cdot q(p)$ no período out-of-time entre 3 estratégias:
1. **Baseline Heurístico:** Preço = $\text{Mediana do Concorrente} \times 0.99$.
2. **Modelo Estimado (Copiloto):** Preço = $c \cdot \frac{\hat{\eta}}{1 + \hat{\eta}}$.
3. **Ótimo Teórico:** Preço = $c \cdot \frac{\eta_{\text{real}}}{1 + \eta_{\text{real}}}$.

In [ ]:
profit_results = []

for vertical, config in VERTICAL_CONFIGS.items():
    sub_test = df_test[df_test["vertical"] == vertical].copy()
    eta_hat = estimated_params[vertical]
    eta_real = config.true_elasticity
    
    # 1. Baseline Heurístico
    p_baseline = sub_test["competitor_median_price"] * 0.99
    
    # 2. Modelo Estimado com Elasticidade Calibrada
    factor_model = eta_hat / (1.0 + eta_hat)
    p_model = sub_test["cost_price"] * factor_model
    
    # 3. Teórico Real
    factor_true = eta_real / (1.0 + eta_real)
    p_oracle = sub_test["cost_price"] * factor_true
    
    # Função de demanda real simulada para apuração da quantidade realizada:
    # ln(q) = alpha + eta_real * ln(p) + beta_real * ln(p_comp)
    def compute_profit(prices):
        ln_q = (
            config.base_demand_alpha
            + config.true_elasticity * np.log(prices)
            + config.cross_elasticity * np.log(sub_test["competitor_median_price"])
        )
        q = np.maximum(1.0, np.exp(ln_q))
        unit_margin = prices - sub_test["cost_price"]
        return float(np.sum(unit_margin * q))

    profit_base = compute_profit(p_baseline)
    profit_mod = compute_profit(p_model)
    profit_ora = compute_profit(p_oracle)
    
    uplift = ((profit_mod - profit_base) / profit_base) * 100.0
    efficiency = (profit_mod / profit_ora) * 100.0
    
    profit_results.append([
        vertical,
        f"R$ {profit_base:,.2f}",
        f"R$ {profit_mod:,.2f}",
        f"R$ {profit_ora:,.2f}",
        f"{uplift:+.2f} %",
        f"{efficiency:.2f} %"
    ])

print(tabulate(
    profit_results,
    headers=["Vertical", "Lucro Baseline", "Lucro Modelo", "Lucro Ótimo Teórico", "Uplift vs Baseline", "Eficiência vs Ótimo"],
    tablefmt="github"
))